# Data Preparation for YOLO-Based Pediatric Wrist Fracture Detection

This notebook covers the preparation of the GrazPedWri-DX dataset for YOLO-based fracture detection.

The main preprocessing steps include:

1. **Patient-level data splitting:** All images belonging to patients with hidden fractures (773 images) are assigned to the test set, together with all other X-ray images of those patients. The remaining data is divided into training, validation, and test sets, ensuring that images from the same patient never appear in different splits.
2. **Label filtering and remapping:** Only class 3 (fracture) is retained from the original YOLO annotations and remapped to class 0, resulting in a single-class detection task.
3. **YOLO dataset organization:** Images and their corresponding annotation files are copied into the appropriate directory structure.
4. **Dataset validation:** The resulting splits are inspected to verify the number of images and annotations, including empty annotation files.

## 1. Import Libraries

In [1]:
import os
import shutil

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from tqdm import tqdm

In [2]:
# Load the original dataset
df = pd.read_csv(
    '/kaggle/input/datasets/jasonroggy/grazpedwri-dx/dataset.csv'
)

# Identify patients with hidden fractures
hidden_fracture_condition = (
                                df['ao_classification'].notna()
                                & df['fracture_visible'].isna()
                            )

patients_with_hidden_fractures = (
                                        df.loc[hidden_fracture_condition, 'patient_id'].unique()
                                    )

# Assign all images belonging to these patients to the test set
hidden_patients_test_df = df[
                                df['patient_id'].isin(patients_with_hidden_fractures)
                            ].copy()

# Keep the remaining patients for further splitting
remaining_df = df[
                    ~df['patient_id'].isin(patients_with_hidden_fractures)
                ].copy()

print(f"Patients with hidden fractures: {len(patients_with_hidden_fractures)}")
print(f"Images assigned directly to test: {len(hidden_patients_test_df)}")
print(f"Remaining images: {len(remaining_df)}")

# Split the remaining data into training and temporary sets
# 79% is assigned to training, while 21% is reserved for validation and test
gss1 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.79,
    random_state=42
)

train_idx, temp_idx = next(
    gss1.split(remaining_df, groups=remaining_df['patient_id'])
)

train_df = remaining_df.iloc[train_idx].copy()
temp_df = remaining_df.iloc[temp_idx].copy()

# Split the temporary set into validation and regular test sets
# 70% goes to validation and 30% to test
gss2 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=42
)

val_idx, test_idx = next(
    gss2.split(temp_df, groups=temp_df['patient_id'])
)

val_df = temp_df.iloc[val_idx].copy()
regular_test_df = temp_df.iloc[test_idx].copy()

# Mark images containing hidden fractures in the test set
hidden_patients_test_df['is_hidden'] = (
    hidden_fracture_condition.loc[hidden_patients_test_df.index]
    .astype(int)
)

regular_test_df['is_hidden'] = 0

# Combine regular test images with images from patients with hidden fractures
final_test_df = pd.concat(
    [regular_test_df, hidden_patients_test_df],
    ignore_index=True
)

# Verify that no patient appears in multiple splits
train_patients = set(train_df['patient_id'])
val_patients = set(val_df['patient_id'])
test_patients = set(final_test_df['patient_id'])

assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

print("Patient-level split verification passed!")

# Display final split statistics
total_images = len(df)

print("\n--- DATASET SPLIT SUMMARY ---")
print(f"Training set:   {len(train_df)} images ({len(train_df) / total_images * 100:.1f}%)")
print(f"Validation set: {len(val_df)} images ({len(val_df) / total_images * 100:.1f}%)")
print(f"Test set:       {len(final_test_df)} images ({len(final_test_df) / total_images * 100:.1f}%)")
print(f"Hidden fractures in test set: {final_test_df['is_hidden'].sum()}")

# Save the resulting splits
train_df.to_csv('train_split.csv', index=False)
val_df.to_csv('val_split.csv', index=False)
final_test_df.to_csv('test_split.csv', index=False)

display(train_df.head(2))

Patients with hidden fractures: 630
Images assigned directly to test: 2254
Remaining images: 18073
Patient-level split verification passed!

--- DATASET SPLIT SUMMARY ---
Training set:   14182 images (69.8%)
Validation set: 2735 images (13.5%)
Test set:       3410 images (16.8%)
Hidden fractures in test set: 773


,filestem,patient_id,study_number,timehash,gender,age,laterality,projection,initial_exam,ao_classification,cast,diagnosis_uncertain,osteopenia,fracture_visible,metal,pixel_spacing,device_manufacturer
4,0003_0662359226_01_WRI-R1_M011,3,1,662359226,M,11.1,R,1,1.0,23-M/3.1,NaN,NaN,NaN,1.0,NaN,0.144,Siemens
5,0003_0662359351_01_WRI-R2_M011,3,1,662359351,M,11.1,R,2,1.0,23-M/3.1,NaN,NaN,NaN,1.0,NaN,0.144,Siemens


In [3]:
# Source directories containing the original images
SOURCE_IMG_DIRS = [
    '/kaggle/input/datasets/jasonroggy/grazpedwri-dx/images_part1',
    '/kaggle/input/datasets/jasonroggy/grazpedwri-dx/images_part2',
    '/kaggle/input/datasets/jasonroggy/grazpedwri-dx/images_part3',
    '/kaggle/input/datasets/jasonroggy/grazpedwri-dx/images_part4'
]

# Source directory containing the original YOLO annotations
SOURCE_LABEL_DIR = (
    '/kaggle/input/datasets/jasonroggy/grazpedwri-dx/'
    'folder_structure/yolov5/labels'
)

# Output directory for the prepared YOLO dataset
OUTPUT_BASE_DIR = '/kaggle/working/yolo_dataset'

# Load the previously created dataset splits
train_df = pd.read_csv('/kaggle/working/train_split.csv')
val_df = pd.read_csv('/kaggle/working/val_split.csv')
test_df = pd.read_csv('/kaggle/working/test_split.csv')


# Create the directory structure required by YOLO
splits = ['train', 'val', 'test']

for split in splits:
    os.makedirs(
        os.path.join(OUTPUT_BASE_DIR, 'images', split),
        exist_ok=True
    )
    os.makedirs(
        os.path.join(OUTPUT_BASE_DIR, 'labels', split),
        exist_ok=True
    )

# Locate an image across the source directories
def find_image(file_stem):
    for folder in SOURCE_IMG_DIRS:
        image_path = os.path.join(folder, f"{file_stem}.png")

        if os.path.exists(image_path):
            return image_path, f"{file_stem}.png"

    return None, None

# Create one YOLO dataset split
def create_split(dataframe, split_name):
    print(f"\nCreating {split_name} split...")

    for _, row in tqdm(dataframe.iterrows(), total=len(dataframe)):
        file_stem = row['filestem']

        # Locate and copy the corresponding image
        source_image_path, image_name = find_image(file_stem)

        if source_image_path is not None:
            destination_image_path = os.path.join(
                OUTPUT_BASE_DIR, 'images', split_name, image_name
            )
            shutil.copy(source_image_path, destination_image_path)
        else:
            print(f"Warning: Image {file_stem} was not found.")

        # Filter annotations and remap fracture class (3) to class (0)
        source_label_path = os.path.join(
            SOURCE_LABEL_DIR, f"{file_stem}.txt"
        )

        destination_label_path = os.path.join(
            OUTPUT_BASE_DIR, 'labels', split_name, f"{file_stem}.txt"
        )

        filtered_annotations = []

        if os.path.exists(source_label_path):
            with open(source_label_path, 'r', encoding='utf-8') as file:
                annotations = file.readlines()

            for annotation in annotations:
                parts = annotation.strip().split()

                if parts and parts[0] == '3':
                    parts[0] = '0'
                    filtered_annotations.append(" ".join(parts) + "\n")

        # Empty annotation files represent images without visible fractures
        with open(destination_label_path, 'w', encoding='utf-8') as file:
            file.writelines(filtered_annotations)

    print(f"{split_name.capitalize()} split created.")

# Generate all three dataset splits
create_split(train_df, 'train')
create_split(val_df, 'val')
create_split(test_df, 'test')

print("YOLO dataset preparation completed!")


Creating train split...


100%|██████████| 14182/14182 [27:55<00:00,  8.46it/s]


Train split created.

Creating val split...


100%|██████████| 2735/2735 [07:27<00:00,  6.11it/s]


Val split created.

Creating test split...


100%|██████████| 3410/3410 [09:20<00:00,  6.08it/s]

Test split created.
YOLO dataset preparation completed!


In [4]:
BASE_DIR = OUTPUT_BASE_DIR
splits = ['train', 'val', 'test']

print("--- YOLO DATASET STATISTICS ---")

for split in splits:
    image_dir = os.path.join(BASE_DIR, 'images', split)
    label_dir = os.path.join(BASE_DIR, 'labels', split)

    # Count images and annotation files
    num_images = len(os.listdir(image_dir))
    label_files = os.listdir(label_dir)
    num_labels = len(label_files)

    # Count empty annotation files
    num_empty_labels = sum(
        os.path.getsize(os.path.join(label_dir, label_file)) == 0
        for label_file in label_files
    )

    num_visible_fractures = num_labels - num_empty_labels

    # Check whether every image has a corresponding annotation file
    image_stems = {
        os.path.splitext(file)[0]
        for file in os.listdir(image_dir)
    }

    label_stems = {
        os.path.splitext(file)[0]
        for file in label_files
    }

    missing_labels = image_stems - label_stems
    missing_images = label_stems - image_stems

    print(f"\n[{split.upper()} SPLIT]")
    print(f"Total images: {num_images}")
    print(f"Total annotation files: {num_labels}")
    print(f"Images with visible fractures (class 0): {num_visible_fractures}")
    print(f"Empty annotation files: {num_empty_labels}")
    print(f"Images without annotation files: {len(missing_labels)}")
    print(f"Annotation files without images: {len(missing_images)}")

print("\n--------------------------------")

--- YOLO DATASET STATISTICS ---

[TRAIN SPLIT]
Total images: 14182
Total annotation files: 14182
Images with visible fractures (class 0): 9426
Empty annotation files: 4756
Images without annotation files: 0
Annotation files without images: 0

[VAL SPLIT]
Total images: 2735
Total annotation files: 2735
Images with visible fractures (class 0): 1936
Empty annotation files: 799
Images without annotation files: 0
Annotation files without images: 0

[TEST SPLIT]
Total images: 3410
Total annotation files: 3410
Images with visible fractures (class 0): 2188
Empty annotation files: 1222
Images without annotation files: 0
Annotation files without images: 0

--------------------------------
